# Apache Iceberg E-Commerce Dataset Generator & Loader (MinIO + PySpark)

This notebook generates synthetic **Customers** (single CSV) and **Orders** (single ~128 MB Parquet spanning exactly 2 days) datasets directly onto your Docker **MinIO S3** bucket (`s3a://warehouse/raw_data/`), verifies the file sizes and date distribution, and loads them into your Apache Iceberg tables (`demo.ecommerce.customers` and `demo.ecommerce.orders`).

### Prerequisites
- Run this notebook inside your `spark-iceberg` Jupyter environment (`http://localhost:8888`).
- Ensure `spark-defaults.conf` has your MinIO S3A credentials (`s3a://warehouse/`) and the Hadoop AWS JARs (`org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262`) configured.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Inherits MinIO S3A and Iceberg JDBC catalog configs from spark-defaults.conf
spark = SparkSession.builder.appName('Generate-Ecommerce-Datasets').getOrCreate()

# --- Dataset Configuration ---
NUM_CUSTOMERS = 100_000
NUM_ORDERS = 8_000_000               # ~8M rows yields ~120-135 MB in a single Snappy Parquet file
START_DATE = '2026-09-10 00:00:00'
TWO_DAYS_SECONDS = 2 * 24 * 60 * 60  # 172,800 seconds (2026-09-10 and 2026-09-11 only)

CUSTOMERS_CSV_PATH = 's3a://warehouse/raw_data/customers_csv'
ORDERS_PARQUET_PATH = 's3a://warehouse/raw_data/orders_phase1_parquet'

print('Spark Session initialized successfully.')

Spark Session initialized successfully.


26/11/24 01:14:31 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


## 1. Create Namespace & Iceberg Tables

- `demo.ecommerce.customers`: Unpartitioned dimension table.
- `demo.ecommerce.orders`: Fact table using Iceberg **Hidden Partitioning** (`days(order_timestamp)`) and **Merge-on-Read (MoR)** properties, ready for future schema and partition evolution.

In [2]:
spark.sql('CREATE NAMESPACE IF NOT EXISTS demo.ecommerce')

spark.sql('''
CREATE TABLE IF NOT EXISTS demo.ecommerce.customers (
    customer_id BIGINT,
    customer_name STRING,
    segment STRING,
    signup_date DATE
)
USING iceberg
TBLPROPERTIES (
    'format-version' = '2',
    'write.parquet.compression-codec' = 'zstd'
)
''')

spark.sql('''
CREATE TABLE IF NOT EXISTS demo.ecommerce.orders (
    order_id BIGINT,
    customer_id BIGINT,
    order_timestamp TIMESTAMP,
    status STRING,
    total_amount DECIMAL(10, 2)
)
USING iceberg
PARTITIONED BY (days(order_timestamp))
TBLPROPERTIES (
    'format-version' = '2',
    'write.update.mode' = 'merge-on-read',
    'write.delete.mode' = 'merge-on-read',
    'write.merge.mode' = 'merge-on-read',
    'write.parquet.compression-codec' = 'zstd'
)
''')

print('Namespace and Iceberg tables are ready.')

Namespace and Iceberg tables are ready.


## 2. Generate Customers Dataset (Single CSV File)

Generates 100,000 customer records using native Spark SQL expressions and writes a single `.csv` file with headers to MinIO.

In [3]:
segments = F.array(
    F.lit('Retail'), F.lit('Wholesale'), F.lit('Enterprise'), F.lit('SMB')
)

customers_df = (
    spark.range(1, NUM_CUSTOMERS + 1, numPartitions=4)
    .select(
        F.col('id').cast('bigint').alias('customer_id'),
        F.concat(F.lit('Customer_'), F.col('id')).alias('customer_name'),
        F.element_at(segments, (F.floor(F.rand(10) * 4) + 1).cast('int')).alias('segment'),
        F.date_add(
            F.lit('2023-01-01').cast('date'),
            F.floor(F.rand(20) * 1000).cast('int')
        ).alias('signup_date'),
    )
)

(
    customers_df
    .coalesce(1)  # Writes a single part-*.csv file
    .write
    .mode('overwrite')
    .option('header', 'true')
    .csv(CUSTOMERS_CSV_PATH)
)

print(f'Customers CSV written to: {CUSTOMERS_CSV_PATH}')

26/11/24 01:14:44 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/11/24 01:14:45 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/11/24 01:14:45 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


Customers CSV written to: s3a://warehouse/raw_data/customers_csv


## 3. Generate Orders Dataset (Single ~128 MB Parquet File, 2 Days Only)

Generates 8,000,000 order records distributed across **2026-09-10** and **2026-09-11** only, streaming into a single Snappy-compressed Parquet file (~128 MB).

In [4]:
statuses = F.array(
    F.lit('PENDING'), F.lit('PROCESSING'), F.lit('COMPLETED'), F.lit('CANCELLED')
)

start_epoch = F.unix_timestamp(F.lit(START_DATE))

orders_df = (
    spark.range(1, NUM_ORDERS + 1, numPartitions=8)
    .select(
        F.col('id').cast('bigint').alias('order_id'),
        (F.floor(F.rand(1) * NUM_CUSTOMERS) + 1).cast('bigint').alias('customer_id'),
        F.timestamp_seconds(
            start_epoch + F.floor(F.rand(2) * TWO_DAYS_SECONDS).cast('bigint')
        ).alias('order_timestamp'),
        F.element_at(statuses, (F.floor(F.rand(3) * 4) + 1).cast('int')).alias('status'),
        F.round(F.lit(10.0) + (F.rand(4) * 4990.0), 2)
         .cast('decimal(10, 2)')
         .alias('total_amount'),
    )
)

(
    orders_df
    .coalesce(1)  # Streams into a single part-*.parquet file without a full shuffle
    .write
    .mode('overwrite')
    .option('compression', 'snappy')
    .parquet(ORDERS_PARQUET_PATH)
)

print(f'Orders Parquet written to: {ORDERS_PARQUET_PATH}')

Orders Parquet written to: s3a://warehouse/raw_data/orders_phase1_parquet


## 4. Verify Physical File Sizes & 2-Day Date Distribution

Inspect the exact file sizes on MinIO (adjust `NUM_ORDERS` above if you want to fine-tune closer to 128 MB) and confirm only 2 dates exist in `order_timestamp`.

In [ ]:
jvm = spark._jvm
hconf = spark._jsc.hadoopConfiguration()

for label, s3_path in [('Customers CSV', CUSTOMERS_CSV_PATH), ('Orders Parquet', ORDERS_PARQUET_PATH)]:
    path = jvm.org.apache.hadoop.fs.Path(s3_path)
    fs = path.getFileSystem(hconf)
    for status in fs.listStatus(path):
        name = status.getPath().getName()
        if name.endswith(('.parquet', '.csv')):
            size_mb = status.getLen() / (1024 * 1024)
            print(f'{label} -> {name} | Size: {size_mb:.2f} MB')

print('\n--- Orders Count by Date (Expected: 2 Days Only) ---')
(
    spark.read.parquet(ORDERS_PARQUET_PATH)
    .groupBy(F.to_date('order_timestamp').alias('order_date'))
    .count()
    .orderBy('order_date')
    .show()
)

## 5. Load Generated Files into Iceberg Tables

- `customers` CSV is read with explicit type casting before appending.
- `orders` Parquet is appended directly; Iceberg automatically splits the single ~128 MB file into two hidden daily partitions (`2026-09-10` and `2026-09-11`).

In [5]:
# 1. Load Customers CSV into Iceberg
cust_in = (
    spark.read.option('header', 'true').csv(CUSTOMERS_CSV_PATH)
    .select(
        F.col('customer_id').cast('bigint'),
        F.col('customer_name').cast('string'),
        F.col('segment').cast('string'),
        F.col('signup_date').cast('date'),
    )
)
cust_in.writeTo('demo.ecommerce.customers').append()
print('Loaded demo.ecommerce.customers.')

# 2. Load Orders Parquet into Iceberg
orders_in = spark.read.parquet(ORDERS_PARQUET_PATH)
orders_in.writeTo('demo.ecommerce.orders').append()
print('Loaded demo.ecommerce.orders.')

Loaded demo.ecommerce.customers.


[Stage 7:>                                                          (0 + 2) / 2]

Loaded demo.ecommerce.orders.


## 6. Verify Iceberg Tables, Hidden Partitions & Join

Inspect the Iceberg `.files` metadata table to confirm hidden daily partitioning and run a sample join between `orders` and `customers`.

In [6]:
print('--- Iceberg Orders Physical Files & Partitions ---')
spark.sql('''
    SELECT file_path, spec_id, partition, record_count, ROUND(file_size_in_bytes / (1024*1024), 2) AS size_mb
    FROM demo.ecommerce.orders.files
''').show(truncate=False)

--- Iceberg Orders Physical Files & Partitions ---
+----------------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|file_path                                                                                                                               |spec_id|partition   |record_count|size_mb|
+----------------------------------------------------------------------------------------------------------------------------------------+-------+------------+------------+-------+
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-10/00000-33-f6aa7fa3-49ea-4aff-8d50-92919c8ccf56-00001.parquet        |0      |{2026-09-10}|1           |0.0    |
|s3a://warehouse/ecommerce/orders/data/order_timestamp_day=2026-09-11/00000-33-f6aa7fa3-49ea-4aff-8d50-92919c8ccf56-00002.parquet        |0      |{2026-09-11}|1           |0.0    |
|s3a://warehouse/ecommerce/orders/data/order

In [7]:
print('--- Sample Join: Revenue by Customer Segment & Order Date ---')
spark.sql('''
    SELECT
        DATE(o.order_timestamp) AS order_date,
        c.segment,
        COUNT(o.order_id) AS total_orders,
        ROUND(SUM(o.total_amount), 2) AS total_revenue
    FROM demo.ecommerce.orders o
    JOIN demo.ecommerce.customers c
        ON o.customer_id = c.customer_id
    GROUP BY DATE(o.order_timestamp), c.segment
    ORDER BY order_date, c.segment
''').show()

--- Sample Join: Revenue by Customer Segment & Order Date ---


[Stage 11:==================================>                       (3 + 2) / 5]

+----------+----------+------------+-------------+
|order_date|   segment|total_orders|total_revenue|
+----------+----------+------------+-------------+
|2026-09-10|Enterprise|      990211|2477948008.82|
|2026-09-10|    Retail|      995932|2494811809.36|
|2026-09-10|       SMB|      999067|2503932923.40|
|2026-09-10| Wholesale|     1013248|2537522892.34|
|2026-09-11|Enterprise|      991179|2481286052.94|
|2026-09-11|    Retail|      996454|2498669041.62|
|2026-09-11|       SMB|      999310|2505568695.34|
|2026-09-11| Wholesale|     1014599|2542042070.79|
|2026-09-12| Wholesale|           1|      1045.00|
+----------+----------+------------+-------------+

